In [ ]:
---
title: BlueFlux Modeled Daily CO2 and CH4 Wetland Fluxes for Southern Florida
description: Documentation of data transformation
author: Paridhi Parajuli
date: July 15, 2025
execute:
  freeze: true
---

In [ ]:
# Import required libraries
import xarray as xr
import os
import pandas as pd

In [ ]:
### Transform CH4 data
# Ensure the output directory for CH4 exists
os.makedirs("output_ch4", exist_ok=True)

# 🔹 Load the CH₄ dataset
ds = xr.open_dataset("blueflux_fch4_nmol_500m_mean_v1.nc")

# Select the data variable containing CH₄ flux mean values
da = ds["fch4_mean"]

# Get all available time values in the dataset
all_times = da.time.values

# Filter time values from 2000-02-24 onwards
times = all_times[all_times >= pd.Timestamp("2000-02-24")]

# Print how many timesteps are in total vs filtered
print("Total timesteps:", len(all_times), "| Timesteps after 2000-02-24:", len(times))

# Loop through each filtered time and save as a COG
for t in times:
    # Select data for the specific timestep
    da_temp = da.sel(time=t)
    
    # Set output filename
    filename = f"output_ch4/blueflux_fch4mean_v1_{str(t)[:10]}.tif"
    
    # Fill missing data with -9999
    da_temp = da_temp.fillna(-9999)
    
    # Set nodata value
    da_temp.rio.write_nodata(-9999, inplace=True)
    
    # Define the CRS (Coordinate Reference System)
    da_temp.rio.write_crs("epsg:4326", inplace=True)
    
    # Set spatial dimensions
    da_temp.rio.set_spatial_dims("lon", "lat", inplace=True)
    
    # Export to Cloud Optimized GeoTIFF
    da_temp.rio.to_raster(filename, compress="deflate", nodata=-9999, driver="COG")

print("✅ CH₄ COGs Saved!")


In [ ]:

### Transform CO2 data
# Create output directory if it doesn't exist
os.makedirs("output_co2", exist_ok=True)

# 🔹 Load the CO₂ NetCDF dataset
ds = xr.open_dataset("blueflux_fco2_micromol_500m_mean_v1.nc")

# Select the CO₂ flux variable from the dataset
da = ds["fco2_mean"]

# Get all available time values
all_times = da.time.values

# Filter time values from 2000-02-24 onwards
times = all_times[all_times >= pd.Timestamp("2000-02-24")]

# Loop through each selected time slice
for t in times:
    # Select data for the current timestep
    da_temp = da.sel(time=t)
    
    # Define the output filename using the date
    filename = f"output_co2/blueflux_fco2mean_v1_{str(t)[:10]}.tif"
    
    # Replace missing values with -9999
    da_temp = da_temp.fillna(-9999)
    
    # Assign no-data value
    da_temp.rio.write_nodata(-9999, inplace=True)
    
    # Set the coordinate reference system to EPSG:4326
    da_temp.rio.write_crs("epsg:4326", inplace=True)
    
    # Set spatial dimension names
    da_temp.rio.set_spatial_dims("lon", "lat", inplace=True)
    
    #  Export the raster as a Cloud Optimized GeoTIFF
    da_temp.rio.to_raster(filename, compress="deflate", nodata=-9999, driver="COG")

print("✅ CO₂ COGs saved successfully!")
